# /geotag — Endpoint Evaluation

Checks city detection accuracy, geo-scope classification, and street span extraction.

**Prerequisite:** NLP service running. `/readyz` → 200.

In [ ]:
import sys, time, requests
sys.path.insert(0, '.')
from _scorecard import load_fixture, print_scorecard, NLP_BASE_URL, HEADERS

cases = load_fixture('geotag_cases.json')
print(f'Loaded {len(cases)} test cases')

In [ ]:
r = requests.get(f'{NLP_BASE_URL}/readyz', headers=HEADERS)
assert r.status_code == 200, f'Service not ready: {r.status_code} {r.text}'
print('Service ready')

In [ ]:
results = []

for case in cases:
    t0 = time.monotonic()
    resp = requests.post(
        f'{NLP_BASE_URL}/geotag',
        json={
            'article_id': case['article_id'],
            'text':       case['text'],
            'headline':   case.get('headline', ''),
            'source':     case.get('source', ''),
        },
        headers=HEADERS,
    )
    latency = time.monotonic() - t0
    assert resp.status_code == 200, \
        f"{case['article_id']}: HTTP {resp.status_code} — {resp.text}"
    data = resp.json()

    top_city     = data['geo_cities'][0]['city_name'] if data['geo_cities'] else None
    city_ok      = (top_city == case.get('expected_city')) if case.get('expected_city_present') \
                   else (top_city is None or not case.get('expected_city_present', True))
    scope_ok     = data['geo_scope'] == case.get('expected_geo_scope')
    places_ok    = len(data['all_places']) >= case.get('expected_places_min', 0)
    streets_ok   = len(data['geo_streets']) >= case.get('expected_streets_min', 0)
    passed       = city_ok and scope_ok and places_ok and streets_ok
    icon         = '✅' if passed else '❌'

    results.append({
        'id':        case['article_id'],
        'city_ok':   city_ok,
        'scope_ok':  scope_ok,
        'places_ok': places_ok,
        'streets_ok':streets_ok,
        'latency_s': latency,
        'pass':      passed,
    })

    print(f"{icon} [{case['article_id']}]  "
          f"city={top_city!r} (expected {case.get('expected_city')!r})  "
          f"scope={data['geo_scope']!r}  "
          f"places={len(data['all_places'])}  streets={len(data['geo_streets'])}  "
          f"{latency:.2f}s")
    if not passed:
        print(f"   FAIL: city_ok={city_ok}  scope_ok={scope_ok}  "
              f"places_ok={places_ok}  streets_ok={streets_ok}")
    print()

In [ ]:
passing    = [r for r in results if r['pass']]
city_pass  = sum(1 for r in results if r['city_ok'])
scope_pass = sum(1 for r in results if r['scope_ok'])
avg_lat    = sum(r['latency_s'] for r in results) / len(results)

print_scorecard('/geotag', {
    'Cases':                   len(results),
    'Passing (all checks)':    f'{len(passing)}/{len(results)}',
    'City detection accuracy': f'{city_pass}/{len(results)}',
    'Scope accuracy':          f'{scope_pass}/{len(results)}',
    'Avg latency (s)':         avg_lat,
})